Goal. Get reliable, machine-parseable output - the plumbing agents and MCP depend on.
1. Define a JSON Schema for a small structured task (e.g. extract {name, date, amount, category} from 
an email).
2. Use structured outputs / function calling to make the model return exactly that shape.
3. Validate the returned JSON against the schema; on failure, auto-retry with the validation error fed back (max 
N retries).
4. Test on ~10 inputs, including a couple of messy ones.
Deliver: the tool + a short note on failure/retry behaviour

In [ ]:
%uv add google-genai pydantic

In [ ]:
import os
from typing import Literal

from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, ConfigDict

load_dotenv()

GEMINI_KEYS= os.environ['GEMINI_API_KEY']
# MODEL="gemini-3.8-flash"
MODEL="gemini-3.5-flash"

client = genai.Client(api_key=GEMINI_KEYS)

inputs = [
    "Give me the user details from the email 'prit.kanani@kevit.io'",
    "Who is this user prit.kanani@kevit.io?",
    "Give me the weather data for today",
    "Hello, what's your name?",
    "Hey, I am mayur. Can you tell me about my email?",
    "The following is my email. Give me the details about myself: 'kevit.kevit@gmail.com'",
    "Is there any user existing called jayesh@gmail.com?",
    "",
]


class User(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str
    email: str
    gender: Literal["Male", "Female"]


class GetUserDataArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")

    email: str
    
SCHEMA = GetUserDataArgs.model_json_schema()

users = [
    {"name":"krish", "email":"krish@gmail.com", "gender": "Male"},
    {"name":"prit", "email":"prit.kanani@kevit.io","gender": "Male"},
    {"name":"yash", "email":"yash@kevit.io", "gender": "Male"},
    {"name":"monil", "email":"monil@gmail.com", "gender": "Male"},
    {"name":"neha", "email":"neha@gmail.com", "gender": "Female"},
    {"name":"mehul", "email":"mehul@outlook.com", "gender": "Male"},
    {"name":"pruthviraj", "email":"pruthviraj@gmail.com", "gender": "Male"},
    {"name":"jayesh", "email":"jayesh@gmail.com", "gender": "Male"},
]

get_user_data = {
    "type": "function",
    "name": "get_user_data",
    "description": (
        "Look up a user by email address. "
        "Use this function when the user asks for information "
        "about a specific email address or asks whether an email exists."
    ),
    "parameters": SCHEMA,
}

In [ ]:
def user_data_function(**kargs):
    kargs["email"] = kargs["email"].strip().lower()
    
    for user in users:
        if user["email"].strip().lower() == kargs["email"]:
            return {
                "result": True,
                "user": user
            }
    return {
        "result": False,
        "user": user
    }

In [ ]:
from pydantic import ValidationError

max_repeate = 3
prompt = """
You are the user data provider chatbot. you are to provide the user data from email provided to you by calling the function from function discription provided to you

Rules:
- if email is Provided then call for function 
- if email exists then provide the data in tablular formate
- if email doesn't exist then provide the negetive confirmation to the user.
- strictly follow the formation.

User Query: 
"""

def function_interaction(input_text: str, previous_interaction_id: str | None ):
    kwargs = {
            "model": MODEL,
            "input": input_text,
            "response_format": {
                "type": "text",
                "mime_type": "application/json",
                "schema": SCHEMA,
            }
        }
    if previous_interaction_id:
        kwargs["previous_interaction_id"] = previous_interaction_id
        
    return client.interactions.create(**kwargs)

for input in inputs:
    input = prompt + input
    interaction = client.interactions.create(
                model=MODEL,
                input=input,
                tools=[get_user_data],
            )
    function_called = False
    last_error = None
    reattempt_email = ""
    for attempt in range(max_repeate):
        if attempt > 1 and last_error:
            input = f"""
                The previous response failed validation.

                Validation error:
                {last_error}

                Correct the previous JSON response.

                Return ONLY valid JSON matching the schema.
                Do not explain anything.

                Original email:
                {reattempt_email}
                """
        elif attempt > 1 and last_error == None:
            break
        for step in interaction.steps:
            if step.type == "function_call":
                function_called = True
                result = user_data_function(**step.arguments)
                reattempt_email = step.arguments
                intermediate_interaction = function_interaction(input, interaction.previous_interaction_id)
                raw_json = intermediate_interaction.output_text
                try: 
                    result = User.model_dump(raw_json)
                    print(result)
                except ValidationError as exc:
                    last_error = str(exc)

                    print("VALIDATION FAILED:")
                    print(last_error)
    if not function_called:
        print(interaction.output_text)
                    
            

In [ ]:
import json

for query in inputs:
    print("\n" + "=" * 70)
    print("USER:", query)

    try:    
        interaction = client.interactions.create(
            model=MODEL,
            input=query,
            tools=[get_user_data],
        )

        function_called = False
        
        for step in interaction.steps:
            
            if step.type == "function_call":
                print("ARGUMENTS:", step.arguments)
                
                function_called = True
                result = user_data_function(step.arguments["email"])
                
                final_interaction = client.interactions.create(
                    model=MODEL,
                    previous_interaction_id=interaction.id,
                    tools=[get_user_data],
                    input=[{
                        "type": "function_result",
                        "name": step.name,
                        "call_id": step.id,
                        "result": [{"type": "text", "text": json.dumps(result)}]
                    }],
                )
                print("MODEL:", final_interaction.output_text)
        if not function_called:
            print("MODEL:", interaction.output_text)
    except Exception as e:
        print(e)